# Clase en vivo 4 · Orquestación y gobierno de datos · Demo guiada

**Caso:** NexoServicios, una organización ficticia de servicios con un sistema transaccional, un CRM, archivos de terceros y una API.
**Propósito:** integrar una **DAG** con **contratos de calidad**, **responsables** y **criterios de recuperación**.

La demo sigue siempre la misma secuencia de razonamiento:

> **restricción → opción → trade-off → control → evidencia**

No buscamos la arquitectura perfecta, sino una **decisión defendible**, con restricciones explícitas y una forma de detectar cuándo deja de funcionar.

Cómo se usa: corré las celdas **en orden**. Antes de cada una hay una explicación breve, y varias piden una **predicción** antes de correr. Los datos son ficticios y se generan desde el código. No hace falta instalar Airflow: un mini orquestador simula sus conceptos para poder razonar y probar el diseño.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
INSTALAR_XGBOOST = False          # opcional: ponelo en True para sumar XGBoost a la comparación de modelos
if INSTALAR_XGBOOST: REQUISITOS["xgboost"] = "xgboost"

print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")


## Preparación

In [ ]:
# Este archivo es autocontenido: instala solo lo que le falte (se corre una sola vez)
import importlib.util
import subprocess
import sys

LIBRERIAS = ("numpy", "pandas")
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    for extra in ([], ["--user"], ["--break-system-packages"]):          # si un intento falla, prueba el siguiente
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *faltan], capture_output=True, text=True)
        if r.returncode == 0:
            break
    else:
        print("⚠️ No se pudo instalar solo. Corré en una celda:  %pip install " + " ".join(faltan))
        print(r.stderr[-300:])
    importlib.invalidate_caches()
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
print("✅ Librerías listas" if not faltan else f"⚠️ Todavía faltan: {faltan}")

La celda siguiente define el **kit de la clase**: un mini orquestador (DAG, tareas, dependencias, reintentos, backfill), una librería de **controles de calidad** y herramientas de **gobierno** (contratos, enmascarado, matriz RACI). No hace falta leerlo ahora: lo vamos usando.

In [ ]:
# ═══════════════════════════ KIT DE PIPELINES · orquestación, calidad y gobierno ═══════════════════════════
# No hace falta instalar Airflow: este mini orquestador simula sus conceptos principales
# (DAG, tareas, dependencias, programación, reintentos, sensores, backfill) para poder razonar y probar el diseño.
import hashlib
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
from IPython.display import display

# ───────────────────────── 1 · EL ORQUESTADOR ─────────────────────────
class ErrorTransitorio(Exception):
    """Falla que puede resolverse reintentando: un timeout, un límite de tasa."""
class ControlFallido(Exception):
    """Un control bloqueante no pasó: reintentar no lo arregla."""
class SensorVencido(Exception):
    """Lo que se esperaba (un archivo, un dato) no llegó dentro del plazo."""

@dataclass
class Tarea:
    nombre: str
    funcion: object
    depende_de: tuple = ()
    reintentos: int = 2
    espera_s: float = 30.0
    siempre: bool = False          # como trigger_rule="all_done": corre aunque falle algo antes
    descripcion: str = ""

def nuevo_contexto(fecha, almacen=None, **extra):
    """El contexto es lo que viaja por la DAG: la fecha de la corrida, el almacén de destino, los datos y las alertas."""
    return {"fecha": pd.Timestamp(fecha), "almacen": {} if almacen is None else almacen, "datos": {}, "alertas": [], **extra}

class Corrida:
    def __init__(self, dag, fecha):
        self.dag, self.fecha, self.filas = dag, pd.Timestamp(fecha), []
    @property
    def log(self):
        return pd.DataFrame(self.filas, columns=["tarea", "estado", "intentos", "espera simulada (s)", "filas", "detalle"])
    @property
    def exitosa(self):
        return all(f["estado"] in ("exitosa", "alerta") for f in self.filas)
    def estado(self, tarea):
        return next(f["estado"] for f in self.filas if f["tarea"] == tarea)

class Dag:
    """Una DAG: tareas con dependencias, sin ciclos, con una programación y una política de reintentos."""
    def __init__(self, nombre, programacion="0 6 * * *", reintentos=2, espera_s=30.0, responsable="(sin responsable)"):
        self.nombre, self.programacion, self.reintentos, self.espera_s, self.responsable = nombre, programacion, reintentos, espera_s, responsable
        self.tareas = {}

    def tarea(self, nombre=None, depende_de=(), reintentos=None, espera_s=None, siempre=False, descripcion=""):
        """Decorador que registra una función como tarea de la DAG."""
        def registrar(funcion):
            n = nombre or funcion.__name__
            self.tareas[n] = Tarea(n, funcion, tuple(depende_de), self.reintentos if reintentos is None else reintentos,
                                   self.espera_s if espera_s is None else espera_s, siempre, descripcion or (funcion.__doc__ or "").strip())
            return funcion
        return registrar

    def orden(self):
        """Orden de ejecución (orden topológico). Falla si hay una dependencia inexistente o un ciclo."""
        for t in self.tareas.values():
            for d in t.depende_de:
                if d not in self.tareas:
                    raise ValueError(f"La tarea «{t.nombre}» depende de «{d}», que no existe")
        pendientes, listas = dict(self.tareas), []
        while pendientes:
            libres = [n for n, t in pendientes.items() if all(d in listas for d in t.depende_de)]
            if not libres:
                raise ValueError(f"Hay un ciclo entre las tareas: {sorted(pendientes)}")
            listas.append(libres[0]); pendientes.pop(libres[0])
        return listas

    def dibujar(self):
        """Muestra la DAG como niveles: lo que puede correr en paralelo queda en el mismo nivel."""
        nivel = {}
        for n in self.orden():
            nivel[n] = 1 + max([nivel[d] for d in self.tareas[n].depende_de], default=0)
        print(f"DAG «{self.nombre}» · programación: {self.programacion} · responsable: {self.responsable}")
        for k in sorted(set(nivel.values())):
            print(f"  nivel {k}: " + "   ".join(f"[{n}]" + (f" ← {', '.join(self.tareas[n].depende_de)}" if self.tareas[n].depende_de else "") for n in nivel if nivel[n] == k))

    def correr(self, fecha, ctx=None, fallas=None, mostrar=True):
        """Corre la DAG para una fecha. `fallas` inyecta fallas para probar el diseño:
        {"tarea": 2} falla las dos primeras veces; {"tarea": "siempre"} falla siempre."""
        ctx = ctx if ctx is not None else nuevo_contexto(fecha)
        ctx["fecha"] = pd.Timestamp(fecha)
        pendientes = dict(fallas or {})
        corrida = Corrida(self, fecha)
        estados = {}
        for n in self.orden():
            t = self.tareas[n]
            rotas = [d for d in t.depende_de if estados[d] in ("fallida", "omitida")]
            if rotas and not t.siempre:
                estados[n] = "omitida"
                corrida.filas.append({"tarea": n, "estado": "omitida", "intentos": 0, "espera simulada (s)": 0, "filas": None, "detalle": f"no corrió: falló {', '.join(rotas)}"})
                continue
            alertas_antes, espera, estado, detalle, filas, intentos = len(ctx["alertas"]), 0.0, "exitosa", "", None, 0
            for intento in range(1, t.reintentos + 2):
                intentos = intento
                try:
                    if pendientes.get(n) == "siempre" or (isinstance(pendientes.get(n), int) and pendientes[n] > 0):
                        if isinstance(pendientes[n], int):
                            pendientes[n] -= 1
                        raise ErrorTransitorio("falla inyectada (timeout)")
                    resultado = t.funcion(ctx)
                    filas = resultado if isinstance(resultado, (int, np.integer)) else None
                    estado = "alerta" if len(ctx["alertas"]) > alertas_antes else "exitosa"
                    detalle = "; ".join(ctx["alertas"][alertas_antes:])
                    if intento > 1:
                        detalle = f"recuperó en el intento {intento}" + (f"; {detalle}" if detalle else "")
                    break
                except (ControlFallido, SensorVencido) as e:
                    estado, detalle = "fallida", str(e)
                    break
                except Exception as e:
                    detalle = f"{type(e).__name__}: {e}"
                    if intento <= t.reintentos:
                        espera += t.espera_s * 2 ** (intento - 1)         # espera creciente entre reintentos
                    else:
                        estado = "fallida"
            estados[n] = estado
            corrida.filas.append({"tarea": n, "estado": estado, "intentos": intentos, "espera simulada (s)": espera, "filas": filas, "detalle": detalle})
        if mostrar:
            display(corrida.log)
            print(f"Corrida del {pd.Timestamp(fecha).date()}: {'EXITOSA' if corrida.exitosa else 'CON FALLAS'}")
        return corrida

    def backfill(self, desde, hasta, almacen=None, fallas=None):
        """Reprocesa un rango de fechas, una corrida por día. Sirve solo si las tareas son idempotentes."""
        almacen = {} if almacen is None else almacen
        filas = []
        for f in pd.date_range(desde, hasta, freq="D"):
            c = self.correr(f, nuevo_contexto(f, almacen), fallas, mostrar=False)
            filas.append({"fecha": f.date(), "resultado": "exitosa" if c.exitosa else "con fallas", "tareas fallidas": ", ".join(r["tarea"] for r in c.filas if r["estado"] == "fallida")})
        return pd.DataFrame(filas), almacen

# ───────────────────────── 2 · CALIDAD: CONTROLES ─────────────────────────
@dataclass
class Control:
    nombre: str
    dimension: str                # completitud, unicidad, validez, consistencia, frescura, volumen, integridad, seguridad
    regla: object                 # función: tabla -> (pasó, detalle)
    severidad: str = "bloqueante" # "bloqueante" detiene lo que sigue; "no bloqueante" avisa y sigue
    accion: str = ""              # qué se hace cuando no pasa
    responsable: str = ""         # quién decide y actúa

def c_esquema(columnas, **k):
    return Control("esquema esperado", "consistencia", lambda d: (set(columnas) <= set(d.columns), f"faltan {sorted(set(columnas) - set(d.columns))}; sobran {sorted(set(d.columns) - set(columnas))}" if set(columnas) != set(d.columns) else "columnas completas"), **k)
def c_sin_nulos(columnas, max_pct=0.0, **k):
    def regla(d):
        pct = 100 * d[columnas].isna().any(axis=1).mean() if len(d) else 0
        return pct <= max_pct, f"{pct:.1f} % de filas con nulos en {columnas} (máximo {max_pct} %)"
    return Control(f"sin nulos en {', '.join(columnas)}", "completitud", regla, **k)
def c_clave_unica(columna, **k):
    def regla(d):
        rep = int(d[columna].duplicated().sum())
        return rep == 0, f"{rep} valores repetidos en «{columna}»"
    return Control(f"clave única «{columna}»", "unicidad", regla, **k)
def c_rango(columna, minimo, maximo, max_pct=0.0, **k):
    def regla(d):
        fuera = 100 * (~d[columna].between(minimo, maximo)).mean() if len(d) else 0
        return fuera <= max_pct, f"{fuera:.1f} % de «{columna}» fuera de [{minimo}, {maximo}]"
    return Control(f"rango de «{columna}»", "validez", regla, **k)
def c_valores(columna, permitidos, **k):
    def regla(d):
        malos = d.loc[~d[columna].isin(permitidos), columna].unique().tolist()
        return not malos, f"valores no permitidos: {malos}" if malos else "valores válidos"
    return Control(f"valores válidos de «{columna}»", "validez", regla, **k)
def c_frescura(columna_fecha, fecha_esperada, max_dias=0, **k):
    def regla(d):
        ultima = pd.to_datetime(d[columna_fecha]).max() if len(d) else pd.NaT
        ok = pd.notna(ultima) and (pd.Timestamp(fecha_esperada) - ultima).days <= max_dias
        return ok, f"último dato: {ultima.date() if pd.notna(ultima) else 'sin datos'} · esperado: {pd.Timestamp(fecha_esperada).date()}"
    return Control(f"frescura de «{columna_fecha}»", "frescura", regla, **k)
def c_volumen(esperado, tolerancia=0.3, **k):
    def regla(d):
        var = abs(len(d) - esperado) / esperado
        return var <= tolerancia, f"{len(d)} filas contra {esperado} esperadas ({100 * var:.0f} % de diferencia; tolerancia {100 * tolerancia:.0f} %)"
    return Control("volumen razonable", "volumen", regla, **k)
def c_integridad(columna, referencia, **k):
    def regla(d):
        huerfanos = int((~d[columna].isin(referencia)).sum())
        return huerfanos == 0, f"{huerfanos} filas con «{columna}» que no existe en la tabla de referencia"
    return Control(f"integridad de «{columna}»", "integridad", regla, **k)
def c_sin_pii(columnas_sensibles, **k):
    def regla(d):
        expuestas = [c for c in columnas_sensibles if c in d.columns]
        return not expuestas, f"columnas sensibles expuestas: {expuestas}" if expuestas else "sin columnas sensibles"
    return Control("sin datos sensibles expuestos", "seguridad", regla, **k)

def evaluar_controles(tabla, controles, ctx=None, cortar=True):
    """Corre los controles sobre una tabla. Si falla uno BLOQUEANTE y `cortar=True`, levanta ControlFallido
    (la DAG detiene lo que depende de esta tarea). Los NO BLOQUEANTES dejan una alerta y la DAG sigue."""
    filas = []
    for c in controles:
        try:
            ok, detalle = c.regla(tabla)
        except KeyError as e:                                    # si la columna ya no existe, el control falla con un mensaje claro
            ok, detalle = False, f"no se pudo evaluar: falta la columna {e}"
        filas.append({"control": c.nombre, "dimensión": c.dimension, "severidad": c.severidad, "pasó": bool(ok), "detalle": detalle, "acción si falla": c.accion, "responsable": c.responsable})
    informe = pd.DataFrame(filas)
    if ctx is not None:
        ctx.setdefault("controles", []).append(informe)
        for f in filas:
            if not f["pasó"] and f["severidad"] != "bloqueante":
                ctx["alertas"].append(f"{f['control']}: {f['detalle']}")
    bloqueantes = informe[(~informe["pasó"]) & (informe["severidad"] == "bloqueante")]
    if cortar and len(bloqueantes):
        raise ControlFallido("; ".join(f"{r['control']} → {r['detalle']}" for _, r in bloqueantes.iterrows()))
    return informe

# ───────────────────────── 3 · GOBIERNO ─────────────────────────
def enmascarar(tabla, columnas, sal="nexo"):
    """Reemplaza valores sensibles por un código irreversible. El mismo valor da siempre el mismo código, así se puede unir sin exponerlo."""
    t = tabla.copy()
    for c in columnas:
        t[c] = t[c].astype(str).map(lambda v: hashlib.sha256((sal + v).encode()).hexdigest()[:10])
    return t

def contrato_datos(nombre, owner, steward, columnas, frescura_horas, consumidores, controles=()):
    """Un contrato de datos: qué es, quién responde, qué contiene, qué tan fresco debe estar y quién lo consume.
    `columnas` es {nombre: (tipo, clasificación)}, con clasificación pública, interna o sensible."""
    return {"nombre": nombre, "owner": owner, "steward": steward, "columnas": columnas, "frescura_horas": frescura_horas,
            "consumidores": list(consumidores), "controles": [c.nombre for c in controles]}

def ver_contrato(c):
    print(f"CONTRATO «{c['nombre']}» · owner: {c['owner']} · steward: {c['steward']} · frescura máxima: {c['frescura_horas']} h · consumidores: {', '.join(c['consumidores'])}")
    t = pd.DataFrame([{"columna": k, "tipo": v[0], "clasificación": v[1]} for k, v in c["columnas"].items()])
    display(t)
    if c["controles"]:
        print("Controles asociados:", "; ".join(c["controles"]))

def raci(actividades, roles=("Arquitectura", "Operación", "Gobierno", "Revisión")):
    """Matriz RACI: R responsable, A quien rinde cuentas (uno solo por fila), C consultado, I informado.
    `actividades` es {actividad: {rol: "R"/"A"/"A/R"/"C"/"I"}}. Valida que haya exactamente una A por actividad."""
    filas = []
    for act, asignacion in actividades.items():
        cantidad_a = sum("A" in v for v in asignacion.values())
        if cantidad_a != 1:
            raise ValueError(f"«{act}» tiene {cantidad_a} roles con A: debe haber exactamente uno")
        filas.append({"actividad": act, **{r: asignacion.get(r, "") for r in roles}})
    return pd.DataFrame(filas)

# ───────────────────────── 4 · EL CASO NEXOSERVICIOS (datos ficticios) ─────────────────────────
def fuentes_nexo(fecha, n_ordenes=480):
    """Genera las fuentes de NexoServicios para una fecha: operaciones (transaccional), CRM, archivo de un proveedor y API de entregas."""
    fecha = pd.Timestamp(fecha)
    azar = np.random.default_rng(int(fecha.strftime("%Y%m%d")))
    base = np.random.default_rng(2026)                                   # el CRM es el mismo todos los días
    clientes = pd.DataFrame({"id_cliente": [f"C{i:04d}" for i in range(1, 301)],
                             "nombre": [f"Cliente {i:04d}" for i in range(1, 301)],
                             "email": [f"cliente{i:04d}@ejemplo.com" for i in range(1, 301)],
                             "telefono": [f"11-5555-{i:04d}" for i in range(1, 301)],
                             "dni": base.integers(10_000_000, 49_999_999, 300).astype(str),
                             "segmento": base.choice(["A", "B", "C"], 300, p=[.2, .5, .3])})
    n = int(azar.integers(int(n_ordenes * 0.9), int(n_ordenes * 1.1)))
    ordenes = pd.DataFrame({"id_orden": [f"O-{fecha:%Y%m%d}-{i:04d}" for i in range(1, n + 1)], "fecha": fecha.date().isoformat(),
                            "id_cliente": azar.choice(clientes["id_cliente"], n), "canal": azar.choice(["web", "tienda", "telefono"], n, p=[.5, .3, .2]),
                            "monto": np.clip(azar.lognormal(7.0, 0.9, n), 50, 48_000).round(2),
                            "estado": azar.choice(["abierta", "asignada", "cerrada"], n, p=[.15, .25, .6])})
    proveedor = ordenes.sample(frac=0.7, random_state=int(fecha.strftime("%Y%m%d")))[["id_orden"]].assign(costo_proveedor=lambda d: (azar.uniform(0.4, 0.7, len(d)) * ordenes.set_index("id_orden").loc[d["id_orden"], "monto"].to_numpy()).round(2))
    entregas = ordenes[["id_orden"]].assign(estado_entrega=azar.choice(["pendiente", "en camino", "entregada"], n, p=[.2, .3, .5]))
    return {"operaciones": ordenes, "crm": clientes, "proveedor": proveedor.reset_index(drop=True), "entregas": entregas}

# ───────────────────────── 5 · REVISORES: ¿el diseño está completo? ─────────────────────────
def revisar_controles(controles):
    """Marca los controles que registran un fallo pero no dicen qué hacer ni quién decide."""
    filas = []
    for c in controles:
        faltas = []
        if not c.accion.strip():
            faltas.append("no dice qué hacer si falla")
        if not c.responsable.strip():
            faltas.append("no dice quién decide")
        if c.severidad not in ("bloqueante", "no bloqueante"):
            faltas.append("severidad inválida")
        filas.append({"control": c.nombre, "severidad": c.severidad, "estado": "✔ completo" if not faltas else "✘ " + "; ".join(faltas)})
    tabla = pd.DataFrame(filas)
    display(tabla)
    return tabla

def revisar_dag(dag, controles=()):
    """Revisa un diseño de DAG contra una lista corta de criterios de calidad."""
    cumple = {
        "Tiene responsable": dag.responsable != "(sin responsable)",
        "Tiene programación": bool(dag.programacion),
        "Todas las tareas están descritas": all(t.descripcion for t in dag.tareas.values()),
        "Hay al menos un control de calidad entre las tareas": any("control" in n for n in dag.tareas),
        "Hay una tarea que avisa aunque algo falle": any(t.siempre for t in dag.tareas.values()),
        "Los reintentos están acotados (5 o menos)": all(t.reintentos <= 5 for t in dag.tareas.values()),
        "Hay controles bloqueantes y no bloqueantes": {"bloqueante", "no bloqueante"} <= {c.severidad for c in controles},
        "Cada control dice qué hacer y quién decide": all(c.accion.strip() and c.responsable.strip() for c in controles) and len(controles) > 0,
    }
    for criterio, ok in cumple.items():
        print("✔" if ok else "✘", criterio)
    return all(cumple.values())

In [ ]:
# ───────────────────────── LA DAG DE NEXOSERVICIOS ─────────────────────────
COLUMNAS_OPERACIONES = ["id_orden", "fecha", "id_cliente", "canal", "monto", "estado"]
SENSIBLES = ["nombre", "email", "telefono", "dni"]

def controles_entrada(fecha):
    return [c_esquema(COLUMNAS_OPERACIONES, severidad="bloqueante", accion="Frenar y avisar al dueño de la fuente", responsable="Arquitectura"),
            c_clave_unica("id_orden", severidad="bloqueante", accion="Frenar: no se publican cifras con órdenes repetidas", responsable="Arquitectura"),
            c_sin_nulos(["id_cliente", "monto"], severidad="bloqueante", accion="Frenar y reclamar la corrección de la fuente", responsable="Gobierno"),
            c_valores("canal", ["web", "tienda", "telefono"], severidad="no bloqueante", accion="Revisar el canal nuevo y actualizar el catálogo", responsable="Gobierno"),
            c_rango("monto", 1, 50_000, max_pct=1.0, severidad="no bloqueante", accion="Revisar los montos atípicos antes del cierre", responsable="Operación"),
            c_frescura("fecha", fecha, severidad="bloqueante", accion="Frenar: los datos no son de hoy", responsable="Operación"),
            c_volumen(480, 0.3, severidad="no bloqueante", accion="Avisar a Operación y mirar si falta una fuente", responsable="Operación")]

def controles_salida():
    return [c_sin_pii(SENSIBLES, severidad="bloqueante", accion="Frenar: nunca se publican datos sensibles", responsable="Gobierno"),
            c_clave_unica("clave", severidad="bloqueante", accion="Frenar: los indicadores tendrían filas repetidas", responsable="Arquitectura")]

def armar_dag(idempotente=True):
    dag = Dag("nexoservicios_indicadores_diarios", programacion="0 6 * * *", reintentos=2, espera_s=30, responsable="Operación")

    @dag.tarea(descripcion="Lee las operaciones del día del sistema transaccional")
    def extraer_operaciones(ctx):
        ctx["fuentes"] = fuentes_nexo(ctx["fecha"])
        ctx["datos"]["operaciones"] = ctx["fuentes"]["operaciones"].copy()
        if ctx.get("defecto") == "duplicados":                                            # defecto inyectado para la demo
            ctx["datos"]["operaciones"] = pd.concat([ctx["datos"]["operaciones"], ctx["datos"]["operaciones"].head(25)])
        if ctx.get("defecto") == "montos_raros":
            ctx["datos"]["operaciones"].loc[ctx["datos"]["operaciones"].sample(frac=0.05, random_state=1).index, "monto"] = 999_999
        return len(ctx["datos"]["operaciones"])

    @dag.tarea(descripcion="Lee los clientes del CRM")
    def extraer_crm(ctx):
        ctx["datos"]["crm"] = fuentes_nexo(ctx["fecha"])["crm"]
        return len(ctx["datos"]["crm"])

    @dag.tarea(descripcion="Espera el archivo del proveedor (sensor con plazo)")
    def esperar_archivo(ctx):
        if not ctx.get("archivo_llego", True):
            if ctx.get("politica_archivo") == "usar_el_de_ayer":
                ctx["alertas"].append("el archivo no llegó: se usó el del día anterior")
                ctx["datos"]["proveedor"] = fuentes_nexo(ctx["fecha"] - pd.Timedelta(days=1))["proveedor"]
                return len(ctx["datos"]["proveedor"])
            raise SensorVencido("el archivo del proveedor no llegó antes de las 7:00")
        ctx["datos"]["proveedor"] = ctx["fuentes"]["proveedor"] if "fuentes" in ctx else fuentes_nexo(ctx["fecha"])["proveedor"]
        return len(ctx["datos"]["proveedor"])

    @dag.tarea(descripcion="Consulta la API de entregas")
    def llamar_api(ctx):
        ctx["datos"]["entregas"] = fuentes_nexo(ctx["fecha"])["entregas"]
        return len(ctx["datos"]["entregas"])

    @dag.tarea(depende_de=["extraer_operaciones", "extraer_crm", "esperar_archivo", "llamar_api"], reintentos=0,
               descripcion="Controles de calidad sobre lo que entra")
    def controles_entrada_calidad(ctx):
        evaluar_controles(ctx["datos"]["operaciones"], controles_entrada(ctx["fecha"]), ctx)
        evaluar_controles(ctx["datos"]["operaciones"], [c_integridad("id_cliente", ctx["datos"]["crm"]["id_cliente"], severidad="bloqueante", accion="Frenar y revisar el CRM", responsable="Gobierno")], ctx)

    @dag.tarea(depende_de=["controles_entrada_calidad"], descripcion="Une las fuentes, enmascara lo sensible y calcula los indicadores")
    def transformar(ctx):
        d = ctx["datos"]
        crm = enmascarar(d["crm"], SENSIBLES)
        base = d["operaciones"].merge(crm[["id_cliente", "segmento"]], on="id_cliente", how="left").merge(d["proveedor"], on="id_orden", how="left").merge(d["entregas"], on="id_orden", how="left")
        ind = base.groupby("canal").agg(ordenes=("id_orden", "count"), ingresos=("monto", "sum")).reset_index()
        ind.insert(0, "fecha", ctx["fecha"].date().isoformat())
        ind["clave"] = ind["fecha"] + "|" + ind["canal"]
        ctx["datos"]["indicadores"] = ind
        return len(ind)

    @dag.tarea(depende_de=["transformar"], reintentos=0, descripcion="Controles de calidad y seguridad sobre lo que sale")
    def controles_salida_calidad(ctx):
        evaluar_controles(ctx["datos"]["indicadores"], controles_salida(), ctx)

    @dag.tarea(depende_de=["controles_salida_calidad"], descripcion="Carga los indicadores en el destino" + (" (idempotente: reemplaza por clave)" if idempotente else " (ingenua: agrega siempre)"))
    def cargar_indicadores(ctx):
        nuevos = ctx["datos"]["indicadores"]
        actual = ctx["almacen"].get("indicadores", nuevos.iloc[0:0])
        ctx["almacen"]["indicadores"] = (pd.concat([actual, nuevos]).drop_duplicates("clave", keep="last") if idempotente else pd.concat([actual, nuevos])).reset_index(drop=True)
        return len(nuevos)

    @dag.tarea(depende_de=["cargar_indicadores"], reintentos=0, descripcion="Marca el día como publicado para Dirección")
    def publicar(ctx):
        ctx["almacen"].setdefault("publicado", set()).add(ctx["fecha"].date().isoformat())

    @dag.tarea(depende_de=["publicar"], siempre=True, reintentos=0, descripcion="Avisa el resultado, aunque algo haya fallado")
    def avisar(ctx):
        ctx["aviso"] = f"Indicadores del {ctx['fecha'].date()}: " + ("publicados" if ctx["fecha"].date().isoformat() in ctx["almacen"].get("publicado", set()) else "NO publicados")

    return dag

## 1 · El caso incompleto: qué hace falta preguntar

NexoServicios necesita integrar **operaciones, CRM, archivos externos y una API**. Dirección pide **indicadores diarios**; Operaciones necesita **alertas durante el día**; hay **información sensible** y el equipo es **pequeño**.

Antes de diseñar nada, hay preguntas que se tienen que responder. Estas son las cinco familias que **no pueden faltar**:

| Familia | Una pregunta posible |
| --- | --- |
| Volumen o latencia | ¿Cada cuánto se consulta el indicador y en qué ventana de tiempo hay que tenerlo? |
| Seguridad | ¿Qué datos son sensibles y quién puede verlos? |
| Calidad | ¿Qué significa «dato correcto» y cómo se mide? |
| Recuperación | Si un día falla, ¿se vuelve a correr, desde dónde y quién lo decide? |
| Responsabilidad humana | ¿Quién es dueño de cada fuente y quién responde cuando algo falla? |

Una lista centrada solo en productos o tecnologías **todavía no alcanza**: hay que fijar frecuencia, ventana de decisión, clasificación del dato, fuente de verdad, tolerancia a retrasos, reproceso, propiedad y observabilidad.

Mirá las fuentes de un día de datos:

In [ ]:
fuentes = fuentes_nexo("2026-10-15")
for nombre, tabla in fuentes.items():
    print(f"{nombre:<12} {tabla.shape[0]:>4} filas · columnas: {', '.join(tabla.columns)}")
fuentes["crm"].head(3)

Fijate que el CRM trae **nombre, mail, teléfono y DNI**: son datos **sensibles**. Los indicadores de Dirección **no los necesitan**. Lo vamos a retomar en la parte de gobierno.

## 2 · Cómo pensarlo: dos decisiones razonadas

Tomamos dos decisiones del caso y las razonamos con la secuencia **restricción → opción → trade-off → control → evidencia**.

| Paso | Decisión A · ¿Cada cuánto corre el indicador? | Decisión B · ¿Qué hacemos si el archivo del proveedor no llega? |
| --- | --- | --- |
| **Restricción** (comprobable) | Dirección consulta el indicador **una vez por día**, antes de las 8:00 | Los indicadores de hoy **no usan** el archivo del proveedor (solo órdenes e ingresos) |
| **Opciones** | 1) Cada 5 minutos · 2) Una vez por día, a las 6:00 | 1) Bloquear la publicación · 2) Publicar y avisar que falta el archivo |
| **Elegida y costo aceptado** | Una vez por día. Costo: no hay datos intradía | Publicar y avisar. Costo: el aviso tiene que llegar a una persona |
| **Control** | Frescura: el dato es de hoy; alerta si a las 7:30 no está publicado | Alerta **no bloqueante** de archivo faltante |
| **Evidencia** | El registro de la corrida con la hora de publicación | El registro con la alerta, y quién la recibió |

Dos cosas para notar. La decisión B **depende de una restricción**: si mañana los indicadores incluyen el margen, que sí usa el archivo, el control pasa a ser **bloqueante**. Y cada decisión declara **qué se descarta** y **cómo se sabría que dejó de funcionar**.

## 3 · El orquestador: los conceptos de Airflow, en una DAG

Una **DAG** es un grafo dirigido sin ciclos: tareas con **dependencias** que definen el orden. El orquestador decide **cuándo** corre cada una y **qué hacer** cuando algo falla. Estos son los conceptos, con su equivalente en Airflow:

| Concepto | En el kit | En Airflow |
| --- | --- | --- |
| Tarea | `@dag.tarea()` | `@task` o un operador |
| Dependencia | `depende_de=[...]` | `a >> b` |
| Programación | `programacion="0 6 * * *"` | `schedule="0 6 * * *"` |
| Reintentos con espera creciente | `reintentos=2, espera_s=30` | `retries`, `retry_delay`, `retry_exponential_backoff` |
| Sensor (esperar algo) | `SensorVencido` | Sensores, con `timeout` y `poke_interval` |
| Correr aunque algo falle | `siempre=True` | `trigger_rule="all_done"` |
| Reprocesar fechas pasadas | `dag.backfill(...)` | `backfill` y `catchup` |

Armamos la DAG de NexoServicios y la dibujamos. Cada nivel es lo que **puede correr en paralelo**:

In [ ]:
dag = armar_dag()
dag.dibujar()

**Cómo leerla:** las cuatro fuentes se leen en paralelo (nivel 1). Los **controles de calidad de entrada** esperan a las cuatro. Recién después se transforma, se controla lo que **sale**, se carga y se publica. La tarea `avisar` corre **siempre**, aunque algo haya fallado antes.

## 4 · El camino feliz

**Predicción antes de correr:** ¿cuántas filas cargamos en el destino? (Pista: indicadores por canal, un día.)

In [ ]:
almacen = {}
ctx = nuevo_contexto("2026-10-15", almacen)
corrida = dag.correr("2026-10-15", ctx)
print(ctx["aviso"], "· filas en el destino:", len(almacen["indicadores"]))
almacen["indicadores"]

**Cómo leer el registro de la corrida:** cada fila es una tarea, con su **estado** (`exitosa`, `alerta`, `fallida` u `omitida`), los **intentos** y las **filas** que procesó. Es la **evidencia** de lo que pasó: otra persona podría reconstruir la corrida solo con esta tabla. El destino quedó con **3 filas**: un indicador por cada canal.

## 5 · Cuando algo falla

### 5.1 · La API falla dos veces

**Predicción antes de correr:** si la API falla las dos primeras veces, ¿la corrida es exitosa? ¿Cuánto espera el orquestador?

In [ ]:
c = dag.correr("2026-10-16", nuevo_contexto("2026-10-16"), fallas={"llamar_api": 2}, mostrar=False)
display(c.log[c.log["tarea"] == "llamar_api"])
print("¿Corrida exitosa?", c.exitosa)

**Cómo leer el resultado:** la tarea necesitó **3 intentos** y esperó **90 segundos simulados** en total (30 y después 60: la espera **crece**). La corrida es exitosa y el registro dice `recuperó en el intento 3`. Un **reintento** sirve para fallas **transitorias**; no para un dato incorrecto.

### 5.2 · La API se cae del todo

Ahora la API **falla siempre**. Mirá qué pasa con lo que viene después, y con la tarea `avisar`:

In [ ]:
ctx = nuevo_contexto("2026-10-16")
c = dag.correr("2026-10-16", ctx, fallas={"llamar_api": "siempre"}, mostrar=False)
display(c.log[["tarea", "estado", "intentos", "detalle"]])
print(ctx["aviso"])

**Cómo leer el resultado:** después de agotar los reintentos, `llamar_api` queda `fallida` y todo lo que **depende** de ella queda `omitida`: no se publica nada. `avisar` corre igual. Pero ojo: **avisar no es recuperar**. Monitorear dice *qué* pasó; la recuperación define *qué se hace*:

| Etapa | En este caso |
| --- | --- |
| **Detectar** | La tarea `avisar` informa que los indicadores no se publicaron |
| **Contener** | No se publica nada: Dirección no ve cifras incompletas |
| **Recuperar** | Cuando la API vuelve, se reejecuta el día, y se puede porque la carga es idempotente (parte 6) |
| **Decidir** | Operación es la responsable de reejecutar; si pasan las 8:00, avisa a Dirección |

### 5.3 · El archivo del proveedor no llega

Es la decisión B de la parte 2. Probamos las dos opciones:

In [ ]:
for politica in ["bloquear", "usar_el_de_ayer"]:
    ctx = nuevo_contexto("2026-10-15", {})
    ctx["archivo_llego"] = False
    ctx["politica_archivo"] = politica
    c = dag.correr("2026-10-15", ctx, mostrar=False)
    print(f"{politica:<16} → archivo: {c.estado('esperar_archivo'):<8} · {ctx['aviso']} · alertas: {ctx['alertas']}")

**Cómo leer el resultado:** con `bloquear`, el sensor vence y **no se publica nada**. Con `usar_el_de_ayer`, se publica y la tarea queda en `alerta`. No hay una opción «correcta» en abstracto: depende de **si el indicador usa ese archivo**. Hoy no lo usa, así que bloquear sería un costo sin beneficio.

## 6 · El error más común: reprocesar y duplicar

Una carga **idempotente** deja el mismo resultado si se corre una vez o diez. Una carga **ingenua** agrega siempre. Corremos el **mismo día dos veces** con cada una.

**Predicción antes de correr:** ¿cuántas filas queda con cada carga? ¿Y los ingresos del día?

In [ ]:
filas = []
for idempotente in (False, True):
    d = armar_dag(idempotente)
    alm = {}
    for _ in range(2):
        d.correr("2026-10-15", nuevo_contexto("2026-10-15", alm), mostrar=False)
    filas.append({"carga": "idempotente" if idempotente else "ingenua", "filas en el destino": len(alm["indicadores"]),
                  "ingresos del día": round(alm["indicadores"]["ingresos"].sum(), 2)})
pd.DataFrame(filas)

**Cómo leer la tabla:** la carga ingenua deja **6 filas** y los ingresos del día **duplicados** (1.697.302 contra 848.651). Nadie recibe un error: simplemente los números quedan **mal**. La carga idempotente **reemplaza por clave** y queda en 3 filas. Por eso un reintento o un reproceso solo es seguro si la tarea es idempotente.

### Backfill: reprocesar varios días

Una vez que las cargas son idempotentes, se puede **reprocesar un rango** de fechas sin miedo. Corremos siete días **dos veces seguidas**:

In [ ]:
alm = {}
resultado, alm = dag.backfill("2026-10-08", "2026-10-14", alm)
resultado, alm = dag.backfill("2026-10-08", "2026-10-14", alm)
print(resultado["resultado"].value_counts().to_dict())
print(f"Filas en el destino después de dos backfills: {len(alm['indicadores'])} (7 días × 3 canales = {7 * 3})")

## 7 · Calidad: controles bloqueantes y no bloqueantes

La calidad **no es solo «no hay nulos»**. Son varias dimensiones, y cada control tiene una **severidad**:

- **Bloqueante:** detiene todo lo que depende. Se usa cuando seguir **produciría un resultado incorrecto o inseguro**.
- **No bloqueante:** deja una **alerta** y la DAG sigue. Se usa cuando el dato **sirve pero hay que revisarlo**.

Y cada control tiene que decir **qué se hace si falla** y **quién decide**. Estos son los de entrada:

In [ ]:
controles = controles_entrada(pd.Timestamp("2026-10-15"))
pd.DataFrame([{"control": c.nombre, "dimensión": c.dimension, "severidad": c.severidad, "acción si falla": c.accion, "responsable": c.responsable} for c in controles])

Ahora inyectamos dos defectos, uno por vez: **órdenes repetidas** y **montos fuera de rango**.

**Predicción antes de correr:** ¿en cuál de los dos casos sigue la DAG y en cuál se frena?

In [ ]:
for defecto in ["duplicados", "montos_raros"]:
    ctx = nuevo_contexto("2026-10-15", {})
    ctx["defecto"] = defecto
    c = dag.correr("2026-10-15", ctx, mostrar=False)
    print(f"── Defecto «{defecto}» → corrida exitosa: {c.exitosa}")
    display(c.log.loc[c.log["estado"].isin(["fallida", "alerta"]), ["tarea", "estado", "detalle"]])

**Cómo leer el resultado:** con **órdenes repetidas**, el control de clave única es **bloqueante**: la DAG se frena y no se publica. Con **montos fuera de rango** (casi 5 %), el control es **no bloqueante**: la tarea queda en `alerta`, la DAG **sigue**, y alguien tiene que revisar. Elegir la severidad es una **decisión de negocio**, no técnica.

## 8 · Gobierno: contrato, datos sensibles y responsables

El **gobierno de datos** responde: ¿qué es este dato, quién responde por él, quién puede verlo y quién lo consume? Tres herramientas.

### Contrato de datos

Es un acuerdo escrito sobre una tabla. Define **dueño**, **columnas con su clasificación**, **frescura** y **consumidores**:

In [ ]:
contrato = contrato_datos(
    nombre="indicadores_diarios", owner="Dirección Comercial", steward="Gobierno de Datos",
    columnas={"fecha": ("fecha", "pública"), "canal": ("texto", "pública"), "ordenes": ("entero", "interna"), "ingresos": ("decimal", "interna")},
    frescura_horas=26, consumidores=["Dirección", "Finanzas"], controles=controles_salida())
ver_contrato(contrato)

### Datos sensibles: se enmascaran antes de salir

El CRM trae nombre, mail, teléfono y DNI. El indicador no los necesita, así que **se enmascaran** en la transformación y, además, un control **bloqueante** verifica que no salgan. El mismo valor da siempre el mismo código, entonces se puede unir sin exponerlo:

In [ ]:
crm = fuentes["crm"]
pd.concat([crm[["id_cliente", "email", "dni"]].head(3).add_prefix("antes · ").reset_index(drop=True),
           enmascarar(crm, ["email", "dni"])[["email", "dni"]].head(3).add_prefix("después · ").reset_index(drop=True)], axis=1)

### Matriz RACI

Una **matriz RACI** dice, para cada actividad, quién es **R**esponsable (hace), quién **A**prueba o rinde cuentas (**uno solo** por fila), quién es **C**onsultado y quién queda **I**nformado. La función valida que haya **exactamente una A** por actividad:

In [ ]:
matriz = raci({
    "Definir el contrato de datos":                      {"Arquitectura": "R", "Operación": "C", "Gobierno": "A", "Revisión": "I"},
    "Construir y versionar la DAG":                      {"Arquitectura": "A/R", "Operación": "C", "Gobierno": "I", "Revisión": "C"},
    "Clasificar y proteger los datos sensibles":         {"Arquitectura": "C", "Operación": "I", "Gobierno": "A/R", "Revisión": "I"},
    "Decidir qué pasa si un control bloqueante falla":   {"Arquitectura": "C", "Operación": "R", "Gobierno": "A", "Revisión": "I"},
    "Responder alertas y recuperar":                     {"Arquitectura": "C", "Operación": "A/R", "Gobierno": "I", "Revisión": "I"},
    "Aprobar un cambio de esquema":                      {"Arquitectura": "R", "Operación": "C", "Gobierno": "A", "Revisión": "C"},
})
matriz

Probá qué pasa si una actividad **no tiene responsable de rendir cuentas**, o si tiene dos:

In [ ]:
try:
    raci({"Responder alertas": {"Arquitectura": "R", "Operación": "R", "Gobierno": "I", "Revisión": "I"}})
except ValueError as error:
    print("Error esperado:", error)

## 9 · Dos errores intencionales: encontralos

La guía de la clase pide encontrar los errores **antes** de que alguien los explique.

### Error 1 · «Tiempo real» para un indicador que se consulta una vez por día

Alguien propone que la DAG corra **cada 5 minutos**. Dirección consulta el indicador **una vez por día**.

**Predicción antes de correr:** ¿cuántas corridas más por día implica? ¿Y cuántas llamadas a la API?

In [ ]:
corridas = pd.DataFrame({"propuesta": ["Una vez por día (6:00)", "Cada 5 minutos"], "corridas por día": [1, 24 * 60 // 5]})
corridas["llamadas a la API por día"] = corridas["corridas por día"] * 1        # la DAG llama a la API una vez por corrida
corridas["veces más que la necesaria"] = corridas["corridas por día"] / corridas["corridas por día"].min()
corridas

**Cómo leer la tabla:** correr cada 5 minutos son **288 corridas por día** en lugar de una, **288 veces más** llamadas a la API, más carga en las fuentes, más alertas y más cosas que pueden fallar, para un dato que **nadie mira** hasta las 8:00. La frecuencia sale del **requisito**, no de lo que la herramienta permite. Operaciones sí necesita alertas durante el día, pero eso es **otro requisito** y se resuelve con otra DAG, más frecuente y más chica.

### Error 2 · Un control que registra el fallo pero no dice qué hacer

Estos controles **pasan** o **fallan** bien. ¿Cuál es el problema?

In [ ]:
flojos = [c_clave_unica("id_orden", severidad="bloqueante"),
          c_rango("monto", 1, 50_000, severidad="no bloqueante", responsable="Operación"),
          c_valores("canal", ["web", "tienda", "telefono"], severidad="no bloqueante", accion="Revisar el canal nuevo")]
_ = revisar_controles(flojos)

**Cómo leer el resultado:** el primero no dice **qué hacer** ni **quién decide**; el segundo no dice **qué hacer**; el tercero no dice **quién decide**. Un control así **detecta pero no habilita ninguna acción**: la alerta salta y nadie sabe qué corresponde. **Monitorear no es recuperar.**

## 10 · La DAG, revisada con una lista

Antes de dar un diseño por terminado, se revisa contra una lista corta. Es la misma idea de la **revisión cruzada** que vas a usar con tu equipo:

In [ ]:
print("── La DAG de NexoServicios")
_ = revisar_dag(dag, controles_entrada(pd.Timestamp("2026-10-15")) + controles_salida())

print("\n── Un borrador apurado")
borrador = Dag("reporte_ventas", programacion="0 6 * * *")
@borrador.tarea()
def cargar_todo(ctx):
    pass
_ = revisar_dag(borrador)

**Cómo leer el resultado:** la DAG de NexoServicios cumple los ocho criterios. El borrador apurado no tiene **responsable**, ni **descripciones**, ni **controles**, ni una tarea que **avise** aunque falle algo. Todo lo que dice `✘` es una pregunta que el equipo evaluador te va a hacer.

## 11 · Cuándo sí y cuándo no usar un orquestador

El orquestador es una herramienta, no un objetivo. Una regla corta para decidir:

| Pregunta | Si la respuesta es sí | Si es no |
| --- | --- | --- |
| ¿Hay **más de una tarea** con dependencias? | Seguí a la pregunta siguiente | Un `cron` o un script con alerta alcanza |
| ¿Necesita **respuesta continua** en segundos o minutos? | Eso es **procesamiento por eventos o streaming**; el orquestador coordina solo la parte por lotes | Seguí a la pregunta siguiente |
| ¿Hay **reintentos, reprocesos, alertas** o **varios responsables**? | Un orquestador se justifica | Un `cron` con controles alcanza, y se revisa cuando el proceso crezca |

Y una advertencia: **el orquestador coordina, no transforma**. Los cálculos pesados se hacen en la base de datos o en el motor de procesamiento; la DAG solo dice cuándo y en qué orden.

---
## Para llevarte

- **Restricción, opción, trade-off, control, evidencia:** así se razona una decisión.
- Una DAG sin **responsable**, **controles** y **recuperación** es solo un dibujo.
- Un **reintento** sirve para fallas transitorias; una **carga idempotente** hace seguro el reproceso.
- **Bloqueante** o **no bloqueante** es una decisión de negocio, y cada control dice qué hacer y quién decide.
- **Monitorear no es recuperar.**
- Los datos **sensibles** se clasifican, se enmascaran y se controlan **antes de salir**.
- La frecuencia sale del **requisito**, no de la herramienta.